# RankResume - Document to PostgreSQL Pipeline

This notebook runs local document extraction, structured JSON parsing, validation, PostgreSQL JSONB storage, and dataset export. It does not implement matching or ranking.

## 1. Install dependencies if needed

In [29]:
# Run once in the notebook kernel if imports are unavailable.
# %pip install pdfplumber python-docx psycopg2-binary skill-extractor

## 2. Import the modular pipeline

In [30]:
import sys
import importlib
from pathlib import Path

DB_STORE = Path('/home/vedant-2506/Desktop/Python_ML/Project/DB_Store')
if str(DB_STORE) not in sys.path:
    sys.path.insert(0, str(DB_STORE))
import rankresume_pipeline
importlib.reload(rankresume_pipeline)
from rankresume_pipeline import *

print('Configured resumes path:', RESUMES_DIR)
print('Configured JDs path:', JDS_DIR)
print('Resume extensions:', sorted(RESUME_EXTENSIONS))
print('JD extensions:', sorted(JD_EXTENSIONS))

Configured resumes path: /home/vedant-2506/Desktop/Python_ML/Project/Data/resumes
Configured JDs path: /home/vedant-2506/Desktop/Python_ML/Project/Data/jds
Resume extensions: ['.doc', '.docx', '.pdf']
JD extensions: ['.csv', '.doc', '.docx', '.json', '.pdf']


## 3. Verify input folders and local readers

In [31]:
resume_dir, jd_dir = resolve_input_dirs()
print('Using resumes:', resume_dir)
print('Using JDs:', jd_dir)
print('Resume files:', len(files_in(resume_dir)))
print('JD files:', len(files_in(jd_dir)))
print('pdfplumber reader: available')
print('python-docx reader: available')
print('DOC reader: antiword or LibreOffice is checked only when a .doc file is encountered')

Using resumes: /home/vedant-2506/Desktop/Python_ML/Project/Data/resumes
Using JDs: /home/vedant-2506/Desktop/Python_ML/Project/Data/jds
Resume files: 69
JD files: 25
pdfplumber reader: available
python-docx reader: available
DOC reader: antiword or LibreOffice is checked only when a .doc file is encountered


## 5. PostgreSQL safety check and schema setup

`RESET_TABLES` is deliberately false. The pipeline scans all supported files, checks each filename before extraction, inserts only new documents, and never deletes database rows when source files are removed.

In [32]:
sample = 'C++ C# .NET ASP.NET Node.js React.js Next.js CI/CD Spring Boot REST API PostgreSQL Machine Learning JavaScript Kubernetes Apache Kafka'
print('Technology aliases:', normalize_skills(['C++', 'cpp', 'c plus plus', 'JavaScript', 'js', 'Machine Learning', 'ml']))
print('Discovered skills:', discover_skills(sample))
print('Cleaned sample:', clean_text(sample))
print('Extraction helpers are available; process_batch performs duplicate checks before extracting any existing file.')

Technology aliases: ['c++', 'javascript', 'machine learning']
Discovered skills: ['c++', 'c#', '.net', 'node.js', 'react.js', 'next.js', 'ci/cd', 'spring boot', 'restful api', 'postgresql', 'machine learning', 'javascript', 'apache kafka', 'kubernetes']
Cleaned sample: c++ c# .net .net node.js react.js next.js ci/cd spring boot rest api postgresql machine learning javascript kubernetes apache kafka
Extraction helpers are available; process_batch performs duplicate checks before extracting any existing file.


## 5. PostgreSQL safety check and schema setup

`RESET_TABLES` is deliberately false. The pipeline synchronizes application rows to files currently present in the Data folders and preserves the existing database credentials and JSON schema.

In [33]:
RESET_TABLES = False
connection = connect_db()
print('Verified database:', DB_NAME)
print('Existing public tables:', inspect_tables(connection))
create_schema(connection, reset=RESET_TABLES)
seed_dictionary(connection)
esco_csv = find_esco_skills_csv()
esco_count = import_esco_skills(connection, esco_csv)
load_esco_labels(connection)
print('ESCO CSV:', esco_csv)
print('ESCO records:', esco_count)
print('Schema, technology aliases, and ESCO reference data are ready.')

Verified database: rankresume
Existing public tables: ['ESCO_Skills', 'candidate', 'job_description', 'technology_alias', 'technology_dictionary']


2026-09-23 13:00:03,325 INFO: Imported 13939 ESCO skills from /home/vedant-2506/Downloads/ESCO dataset - v1.2.1 - classification - en - csv/skills_en.csv


ESCO CSV: /home/vedant-2506/Downloads/ESCO dataset - v1.2.1 - classification - en - csv/skills_en.csv
ESCO records: 13939
Schema, technology aliases, and ESCO reference data are ready.


## 6. Run the complete batch

Each document is handled independently. A failed or unsupported file is logged and does not stop the remaining batch.

In [34]:
summary = process_batch(connection, reset=False)
print_summary(summary)
print('\n========== SAMPLE RESUME JSON ==========')
resume_dataset = json.loads((DATA_ROOT / 'resumes.json').read_text(encoding='utf-8'))
print(json.dumps(resume_dataset['resumes'][0], indent=2, ensure_ascii=False)[:3000])
print('\n========== SAMPLE JD JSON ==========')
jd_dataset = json.loads((DATA_ROOT / 'jds.json').read_text(encoding='utf-8'))
print(json.dumps(jd_dataset['job_descriptions'][0], indent=2, ensure_ascii=False)[:3000])

2026-09-23 13:00:05,471 INFO: Imported 13939 ESCO skills from /home/vedant-2506/Downloads/ESCO dataset - v1.2.1 - classification - en - csv/skills_en.csv
2026-09-23 13:00:06,272 INFO: Scanning resumes directory: found 69 supported files
2026-09-23 13:00:06,273 INFO: Scanning JDs directory: found 25 supported files
2026-09-23 13:00:06,274 INFO: [1/69] Checking resume: Aarya_Kale.pdf
2026-09-23 13:00:06,277 INFO: Already exists in database. Skipping: Aarya_Kale.pdf
2026-09-23 13:00:06,279 INFO: [2/69] Checking resume: Abhijeet.pdf
2026-09-23 13:00:06,281 INFO: Already exists in database. Skipping: Abhijeet.pdf
2026-09-23 13:00:06,283 INFO: [3/69] Checking resume: Aditi Bhosure Resume.pdf
2026-09-23 13:00:06,285 INFO: Already exists in database. Skipping: Aditi Bhosure Resume.pdf
2026-09-23 13:00:06,287 INFO: [4/69] Checking resume: Adity_Gowari.pdf
2026-09-23 13:00:06,288 INFO: Already exists in database. Skipping: Adity_Gowari.pdf
2026-09-23 13:00:06,290 INFO: [5/69] Checking resume: Ai

INGESTION SUMMARY

Resumes:
    Files found       : 69
    Already in DB     : 69
    Newly inserted    : 0
    Failed            : 0

JDs:
    Files found       : 25
    Already in DB     : 25
    Newly inserted    : 0
    Failed            : 0

Total files checked  : 94
Total new records    : 0
Total skipped        : 94
Total failed         : 0

Database: rankresume
Candidate records: 69
Job Description records: 25

Resume JSON validation: 0 passed, 0 flagged
JD JSON validation: 0 passed, 0 flagged
Datasets: /home/vedant-2506/Desktop/Python_ML/Project/Data/resumes.json and /home/vedant-2506/Desktop/Python_ML/Project/Data/jds.json
ESCO records: 13939

========== SAMPLE RESUME JSON ==========
{
  "resume_file_name": "Aarya_Kale.pdf",
  "candidate_name": "AARYA KALE",
  "email": "kaleaarya704@gmail.com",
  "phone": "8421749236",
  "structured_data": {
    "skill": [
      "html5",
      "css3",
      "react.js",
      "restful api",
      "java",
      "c++",
      "spring boot",
      

## 7. Validate stored values

In [35]:
with connection.cursor() as cursor:
    cursor.execute('SELECT COUNT(*) FROM "ESCO_Skills"')
    print('ESCO_Skills count:', cursor.fetchone()[0])
    cursor.execute('SELECT preferred_label, alt_labels, concept_uri FROM "ESCO_Skills" LIMIT 10')
    for row in cursor.fetchall():
        print(row)
    cursor.execute('SELECT COUNT(*) FROM candidate')
    print('candidate count:', cursor.fetchone()[0])
    cursor.execute('SELECT COUNT(*) FROM job_description')
    print('job_description count:', cursor.fetchone()[0])
    cursor.execute('''
        SELECT candidate_id, candidate_name, email, phone, resume_file_name, structured_data
        FROM candidate ORDER BY candidate_id DESC LIMIT 5
    ''')
    for row in cursor.fetchall():
        print(row)
    cursor.execute('''
        SELECT jd_id, job_title, jd_file_name, structured_data
        FROM job_description ORDER BY jd_id DESC LIMIT 5
    ''')
    for row in cursor.fetchall():
        print(row)
    current_resume_names = {path.name for path in files_in(resume_dir) if path.suffix.lower() in RESUME_EXTENSIONS}
    current_jd_names = {path.name for path in files_in(jd_dir) if path.suffix.lower() in JD_EXTENSIONS}
    cursor.execute('SELECT resume_file_name FROM candidate')
    stored_resume_names = {row[0] for row in cursor.fetchall()}
    cursor.execute('SELECT jd_file_name FROM job_description')
    stored_jd_names = {row[0] for row in cursor.fetchall()}
    print('application resume scope matches Data:', stored_resume_names == current_resume_names)
    print('application JD scope matches Data:', stored_jd_names == current_jd_names)
    cursor.execute('SELECT raw_text = clean_text FROM candidate ORDER BY candidate_id LIMIT 1')
    print('raw_equals_clean:', cursor.fetchone()[0])

ESCO_Skills count: 13939
('sport and exercise medicine', 'sports medicine\nsports injury medicine\nsports injury treatment\nexercise and sports medicine', 'http://data.europa.eu/esco/skill/00506f28-e884-4496-800c-3477c67eb355')
('food plant design', 'food plant designs\nfood plant layouts\ndesign of food plants\nfood plant developments\nfood plant arrangements\ndesign of food plant\nfood plant models', 'http://data.europa.eu/esco/skill/09dd7302-368e-4b3e-b644-7c03a2aae4a2')
('execute advertising for vehicles', 'achieve advertising measures for the vehicles\nexecute advertising measures for the vehicles\nexecute advertising measure for vehicles\nexecute an advertising measures for the vehicles\nenact advertising measures for the vehicles\nundertake advertising measures for the vehicles\nexecute advertising measures for a vehicle\ncarry out advertising measures for the vehicles', 'http://data.europa.eu/esco/skill/09f07c6a-866b-4587-a771-45b0cd613540')
('perform liquid penetrant tests', '

In [36]:
known_jd = jd_dir / 'JD Intern_SpringCT.pdf'
if known_jd.exists():
    known_raw, known_title = extract_jd_text(known_jd)
    known_result = parse_jd(known_raw, known_title or extract_job_title(known_raw))
    expected_meaningful_skills = {
        'unit testing', 'mobile application development', 'code review', 'debugging',
        'version control', 'git', 'agile', 'scrum', 'bug fixing',
        'object-oriented programming', 'data structures', 'algorithms', 'javascript',
        'html5', 'css3', 'java', 'python', 'restful api', 'problem solving',
        'communication', 'collaboration',
    }
    actual_skills = set(known_result['expected_skills'])
    captured = actual_skills & expected_meaningful_skills
    print('Known JD skills:', known_result['expected_skills'])
    print('Known JD missed:', sorted(expected_meaningful_skills - actual_skills))
    print('Known JD unexpected:', sorted(actual_skills - expected_meaningful_skills))
    print('Known JD recall:', round(len(captured) / len(expected_meaningful_skills), 3))
    print('Known JD precision:', round(len(captured) / len(actual_skills), 3) if actual_skills else 0)
else:
    print('Known JD benchmark skipped; file not present:', known_jd)

connection.close()

Known JD skills: ['unit testing', 'mobile application development', 'code review', 'debugging', 'version control', 'git', 'agile', 'scrum', 'bug fixing', 'object-oriented programming', 'data structures', 'algorithms', 'javascript', 'html5', 'css3', 'java', 'python', 'problem solving', 'communication', 'collaboration', 'back-end', 'restful api']
Known JD missed: []
Known JD unexpected: ['back-end']
Known JD recall: 1.0
Known JD precision: 0.955
